# Tulya Semantic Frontier — exact task information, future-task novelty, and grokking

This notebook creates a finite world where the **exact task-sufficient information frontier is known** and compares it with a learned binary bottleneck.

For a task family `Q`, two raw states are equivalent when every task in `Q` gives the same answer. The resulting task signatures define an exact semantic partition. If there are `K` distinct signatures, the zero-error fixed-length lower bound is `ceil(log2 K)` bits.

We then ask four empirical questions: (1) can gradient learning reach that bound, (2) how do observed-state count `n` and optimization steps change attainability, (3) how much extra retained information supports unseen future tasks, and (4) does representation reorganization coincide with grokking-style delayed generalization?

We also keep **task surprise** `-log2 P(task)` separate from **semantic novelty**, the extra task-relevant distinctions a new task forces the representation to retain.

This is a controlled benchmark, not a claim that semantic partitions/equivalence classes themselves are novel.

In [ ]:
import os, json, math, random
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED=7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working/tulya_semantic_frontier'); OUT.mkdir(parents=True, exist_ok=True)
print('torch',torch.__version__,'device',DEVICE)
if torch.cuda.is_available(): print('GPU',torch.cuda.get_device_name(0))

## 1. Exhaustive world, task bank, exact semantic frontier

The default world contains every 12-bit state: 4096 possible worlds. Tasks mix direct queries, relations, parity, threshold, and counting structure.

In [ ]:
N_BITS=12
def all_states(d):
    z=np.arange(2**d,dtype=np.uint32); s=np.arange(d-1,-1,-1,dtype=np.uint32)
    return ((z[:,None]>>s[None,:])&1).astype(np.uint8)
X_np=all_states(N_BITS); X=torch.tensor(X_np,dtype=torch.float32)
def pc(x): return x.sum(1).astype(np.int64)
T={}
for i in range(6): T[f'bit_{i}']=lambda x,i=i:x[:,i].astype(np.uint8)
for a,b in [(0,1),(2,3),(0,5),(1,7),(3,9),(0,11)]: T[f'xor_{a}_{b}']=lambda x,a=a,b=b:(x[:,a]^x[:,b]).astype(np.uint8)
T['parity_all']=lambda x:(pc(x)%2).astype(np.uint8)
T['majority']=lambda x:(pc(x)>x.shape[1]//2).astype(np.uint8)
T['popcount_mod3_zero']=lambda x:(pc(x)%3==0).astype(np.uint8)
T['first_half_parity']=lambda x:(x[:,:x.shape[1]//2].sum(1)%2).astype(np.uint8)
T['second_half_parity']=lambda x:(x[:,x.shape[1]//2:].sum(1)%2).astype(np.uint8)
TASK_NAMES=list(T); Y_np=np.stack([T[n](X_np) for n in TASK_NAMES],1); Y=torch.tensor(Y_np,dtype=torch.float32)

def pstats(y):
    if y.ndim==1:y=y[:,None]
    _,c=np.unique(y,axis=0,return_counts=True); p=c/c.sum(); K=len(c)
    H=float(-(p*np.log2(p)).sum()); B=int(math.ceil(math.log2(K))) if K>1 else 0
    return K,H,B

NESTED=['bit_0','bit_1','xor_2_3','majority','first_half_parity','popcount_mod3_zero','parity_all']
rows=[]
for k in range(1,len(NESTED)+1):
    ids=[TASK_NAMES.index(n) for n in NESTED[:k]]; K,H,B=pstats(Y_np[:,ids])
    rows.append({'task_count':k,'tasks':','.join(NESTED[:k]),'classes':K,'entropy_bits':H,'fixed_bits':B,'raw_bits':N_BITS})
frontier=pd.DataFrame(rows); display(frontier)
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(frontier.task_count,frontier.entropy_bits,'o-',label='semantic entropy'); ax.step(frontier.task_count,frontier.fixed_bits,where='mid',label='zero-error fixed bits'); ax.axhline(N_BITS,ls='--',label='raw lossless bits'); ax.set(xlabel='tasks',ylabel='bits',title='Exact semantic frontier'); ax.grid(alpha=.25); ax.legend(); plt.show()

## 2. Statistical task surprise is not semantic novelty

For current task family `Q`, semantic novelty of adding task `t` is measured here by the entropy refinement `H(S_(Q+t)) - H(S_Q)`. A rare task may add zero new distinctions; a common task may force a major refinement.

In [ ]:
r=np.arange(1,len(TASK_NAMES)+1,dtype=float); pp=1/(r**1.15); pp/=pp.sum(); P=dict(zip(TASK_NAMES,pp))
BASE=['bit_0','bit_1','xor_2_3']; bid=[TASK_NAMES.index(n) for n in BASE]; _,H0,_=pstats(Y_np[:,bid])
nrows=[]
for n in TASK_NAMES:
    if n in BASE: continue
    _,H1,_=pstats(Y_np[:,bid+[TASK_NAMES.index(n)]])
    nrows.append({'task':n,'probability':P[n],'task_surprise_bits':-math.log2(P[n]),'semantic_novelty_bits':H1-H0})
novelty=pd.DataFrame(nrows); display(novelty.sort_values('task_surprise_bits'))
fig,ax=plt.subplots(figsize=(8,6)); ax.scatter(novelty.task_surprise_bits,novelty.semantic_novelty_bits)
for _,z in novelty.iterrows(): ax.annotate(z.task,(z.task_surprise_bits,z.semantic_novelty_bits),xytext=(4,3),textcoords='offset points',fontsize=8)
ax.set(xlabel='task surprise  -log2 P(task)',ylabel='semantic novelty (bits)',title='Rare is not the same as semantically new'); ax.grid(alpha=.25); plt.show()

## 3. Neural learner with a true binary bottleneck

A float hidden layer is not a `B`-bit representation, so the encoder below emits hard 0/1 codes with a straight-through gradient estimator. We train on only `n_train_states` but evaluate on **all 4096 states**, making memorization and generalization distinguishable.

For the learned code we also compute which unseen tasks are **exactly recoverable**: an unseen task is recoverable iff its label is constant inside every learned code class.

In [ ]:
class Net(nn.Module):
    def __init__(self,bits,out,hidden=128):
        super().__init__(); self.enc=nn.Sequential(nn.Linear(N_BITS,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,bits)); self.dec=nn.Sequential(nn.Linear(bits,hidden),nn.ReLU(),nn.Linear(hidden,out))
    def encode(self,x):
        p=torch.sigmoid(self.enc(x)); h=(p>=.5).float(); return h.detach()-p.detach()+p,p
    def forward(self,x):
        z,p=self.encode(x); return self.dec(z),z,p

@torch.no_grad()
def codes(model):
    p=torch.sigmoid(model.enc(X.to(DEVICE))); return (p>=.5).to(torch.uint8).cpu().numpy()
def code_entropy(c):
    _,n=np.unique(c,axis=0,return_counts=True); p=n/n.sum(); return float(-(p*np.log2(p)).sum())
def recoverable(c,labels=Y_np):
    u,inv=np.unique(c,axis=0,return_inverse=True); ans=[]
    for j in range(labels.shape[1]):
        ans.append(all(len(np.unique(labels[inv==i,j]))==1 for i in range(len(u))))
    return np.array(ans,bool),len(u)

@dataclass
class Cfg:
    tasks: tuple
    bottleneck_bits:int
    n_train_states:int=1024
    steps:int=3000
    eval_every:int=100
    lr:float=2e-3
    wd:float=1e-4
    hidden:int=128
    seed:int=0

def run(cfg):
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)
    ids=[TASK_NAMES.index(n) for n in cfg.tasks]; yy=Y[:,ids]; rng=np.random.default_rng(cfg.seed); tr=rng.choice(len(X),min(cfg.n_train_states,len(X)),replace=False); tr=torch.tensor(tr)
    m=Net(cfg.bottleneck_bits,len(ids),cfg.hidden).to(DEVICE); opt=torch.optim.AdamW(m.parameters(),lr=cfg.lr,weight_decay=cfg.wd)
    xt=X[tr].to(DEVICE); yt=yy[tr].to(DEVICE); xa=X.to(DEVICE); ya=yy.to(DEVICE); hist=[]
    for step in range(cfg.steps+1):
        if step:
            m.train(); lo,_,_=m(xt); loss=F.binary_cross_entropy_with_logits(lo,yt); opt.zero_grad(); loss.backward(); opt.step()
        if step%cfg.eval_every==0 or step==cfg.steps:
            m.eval()
            with torch.no_grad():
                a=(m(xt)[0]>=0).float(); b=(m(xa)[0]>=0).float(); ta=float((a==yt).float().mean()); aa=float((b==ya).float().mean())
            c=codes(m); rec,K=recoverable(c); hist.append({'step':step,'train_acc':ta,'all_state_acc':aa,'learned_classes':K,'code_entropy_bits':code_entropy(c),'recoverable_tasks':int(rec.sum())})
    c=codes(m); rec,K=recoverable(c); K0,H0,B0=pstats(Y_np[:,ids])
    summary={'tasks':list(cfg.tasks),'bottleneck_bits':cfg.bottleneck_bits,'n_train_states':cfg.n_train_states,'steps':cfg.steps,'theoretical_classes':K0,'theoretical_entropy_bits':H0,'theoretical_fixed_bits':B0,'learned_classes':K,'learned_code_entropy_bits':code_entropy(c),'final_train_acc':hist[-1]['train_acc'],'final_all_state_acc':hist[-1]['all_state_acc'],'unseen_recoverable':[TASK_NAMES[i] for i,v in enumerate(rec) if v and TASK_NAMES[i] not in cfg.tasks]}
    return m,pd.DataFrame(hist),summary

## 4. Single frontier run

This starts with four tasks and a bottleneck at the exact fixed-length lower bound. If exhaustive accuracy does not reach 1.0, the gap is an optimization/data result, not an information-theoretic impossibility.

In [ ]:
Q=tuple(NESTED[:4]); ids=[TASK_NAMES.index(n) for n in Q]; _,_,B=pstats(Y_np[:,ids]); print('exact fixed-bit bound:',B)
cfg=Cfg(tasks=Q,bottleneck_bits=B,n_train_states=1024,steps=3000,eval_every=100,seed=SEED)
model,hist,summary=run(cfg); print(json.dumps(summary,indent=2)); hist.to_csv(OUT/'demo_trace.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(hist.step,hist.train_acc,label='train'); ax.plot(hist.step,hist.all_state_acc,label='all states'); ax.set(xlabel='step',ylabel='accuracy',ylim=(0,1.02),title='Memorization vs exhaustive generalization'); ax.grid(alpha=.25); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(hist.step,hist.code_entropy_bits,label='learned code entropy'); ax.axhline(summary['theoretical_entropy_bits'],ls='--',label='exact semantic entropy'); ax.set(xlabel='step',ylabel='bits',title='Representation organization'); ax.grid(alpha=.25); ax.legend(); plt.show()

## 5. `n`-state × bottleneck × compute sweep

Turn this on after the single run. It directly tests the continuum discussed in the research notes: number of observed states `n`, retained bits, and learning compute versus exhaustive generalization and future-task coverage.

In [ ]:
RUN_SWEEP=False
if RUN_SWEEP:
    Q=tuple(NESTED[:4]); ids=[TASK_NAMES.index(n) for n in Q]; _,_,B=pstats(Y_np[:,ids]); rows=[]
    for n in [256,512,1024,2048]:
        for bits in sorted(set([max(1,B-1),B,min(N_BITS,B+2)])):
            for steps in [500,1500,4000]:
                _,_,s=run(Cfg(Q,bits,n,steps,max(100,steps//10),seed=0)); rows.append({'n':n,'bits':bits,'steps':steps,'theory_bits':B,'all_state_acc':s['final_all_state_acc'],'code_entropy':s['learned_code_entropy_bits'],'unseen_recoverable':len(s['unseen_recoverable'])})
    sweep=pd.DataFrame(rows); sweep.to_csv(OUT/'n_bits_compute_sweep.csv',index=False); display(sweep)

## 6. Optional grokking-style long run

This asks whether training memorization, exhaustive generalization, and semantic-code organization occur on different timescales. Grokking is an observed regime, not an assumption; a negative run is still informative.

In [ ]:
RUN_LONG=False
if RUN_LONG:
    Q=('parity_all','first_half_parity','second_half_parity'); ids=[TASK_NAMES.index(n) for n in Q]; _,_,B=pstats(Y_np[:,ids]); g=Cfg(Q,max(B,5),768,30000,100,lr=1e-3,wd=1e-2,hidden=192,seed=3); gm,gh,gs=run(g); gh.to_csv(OUT/'grokking_trace.csv',index=False); json.dump(gs,open(OUT/'grokking_summary.json','w'),indent=2)
    fig,ax=plt.subplots(figsize=(9,5)); ax.plot(gh.step,gh.train_acc,label='train'); ax.plot(gh.step,gh.all_state_acc,label='all states'); ax.set(xlabel='step',ylabel='accuracy',title='Grokking-style probe'); ax.grid(alpha=.25); ax.legend(); plt.show()
    fig,ax=plt.subplots(figsize=(9,5)); ax.plot(gh.step,gh.code_entropy_bits,label='learned code entropy'); ax.axhline(gs['theoretical_entropy_bits'],ls='--',label='exact task entropy'); ax.set(xlabel='step',ylabel='bits',title='Does semantic reorganization track generalization?'); ax.grid(alpha=.25); ax.legend(); plt.show()

## 7. Save benchmark tables and interpretation

Interesting outcomes include: an optimization gap at the exact theoretical width; a compute–representation tradeoff; a data–representation tradeoff; a **generality premium** where extra bits improve unseen-task recoverability; and semantic-code reorganization that precedes or coincides with delayed generalization.

Next extensions: optimize against a probability distribution over future tasks; add lossy task error/rate–distortion; replace independent observed states with trajectories; add causal actions; compare learned partitions with the information-lattice refinement structure; and construct a Pāṇinian-style invariance benchmark where many surface forms share one underlying relational task.

In [ ]:
frontier.to_csv(OUT/'exact_frontier.csv',index=False); novelty.to_csv(OUT/'task_surprise_vs_semantic_novelty.csv',index=False)
meta={'seed':SEED,'n_bits':N_BITS,'n_states':len(X_np),'task_names':TASK_NAMES,'nested_tasks':NESTED,'base_tasks':BASE,'task_probabilities':{k:float(v) for k,v in P.items()}}
json.dump(meta,open(OUT/'benchmark_metadata.json','w'),indent=2)
print('saved to',OUT); print([p.name for p in sorted(OUT.iterdir())])